# LLMによる次単語予測と対話

In [ ]:
# @title プログラムを動かすための設定
!pip -q install -U "transformers>=4.37.0" accelerate bitsandbytes sentencepiece matplotlib
!apt-get -y install fonts-noto-cjk

import torch
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_prop = fm.FontProperties(fname=font_path)

from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import display, HTML

In [ ]:
# @title AIモデルのダウンロード
#MODEL_NAME = "Qwen/Qwen2.5-3B"
#MODEL_NAME = "Qwen/Qwen2.5-7B" # 量子化なしでVRAM13GB ダウンロード5～11分
#MODEL_NAME = "Qwen/Qwen2.5-14B" # 量子化必須．
MODEL_NAME = "tokyotech-llm/Llama-3.1-Swallow-8B-v0.5" # 量子化なしでVRAM13GB, ありでVRAM6GB．ダウンロード5分
DO_QUANTIZE = True

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if DO_QUANTIZE:
    from transformers import BitsAndBytesConfig

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quant_config,
        device_map="auto" if torch.cuda.is_available() else None,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype="auto",
        device_map="auto" if torch.cuda.is_available() else None,
    )

if not torch.cuda.is_available():
    model = model.to(device)

model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("loaded:", MODEL_NAME)

In [ ]:
# @title 途中までの文章を与える（文章を初期化）

prompt = """\
おはようござい
"""



prompt = prompt.strip()
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

In [ ]:
# @title 次の単語を生成
@torch.no_grad()
def step_once_for_class(input_ids, top_k=8):
    outputs = model(input_ids=input_ids)
    next_word_scores = outputs.logits[:, -1, :]
    probs = torch.softmax(next_word_scores, dim=-1)

    top_probs, top_ids = torch.topk(probs, k=top_k, dim=-1)

    # 候補の文字列を作る
    candidates = []
    values = []

    for wid, p in zip(top_ids[0], top_probs[0]):
        word = tokenizer.decode([int(wid)], skip_special_tokens=False)
        word = word.replace("\n", "\\n").replace("\t", "\\t")
        candidates.append(word)
        values.append(float(p))

    # 棒グラフを表示
    plt.figure(figsize=(5, 3))
    plt.bar(candidates, values)
    plt.xlabel("次の単語候補", fontproperties=font_prop, fontsize=12)
    plt.ylabel("確率", fontproperties=font_prop, fontsize=12)
    plt.xticks(rotation=45, ha="right", fontproperties=font_prop, fontsize=14)
    plt.tight_layout()
    plt.show()

    # いちばん確率が高いものを選ぶ
    next_word_id = torch.argmax(probs, dim=-1, keepdim=True)
    chosen_word = tokenizer.decode([int(next_word_id[0, 0])], skip_special_tokens=False)

    new_input_ids = torch.cat([input_ids, next_word_id], dim=-1)

    # 以前までの文章
    old_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    # 新しく選ばれた語を足した後の全文
    new_text = tokenizer.decode(new_input_ids[0], skip_special_tokens=True)

    # 赤字表示用: 末尾に追加された部分をなるべくそのまま赤くする
    added_text = new_text[len(old_text):]

    html = f"""
    <div style="font-size: 15px; line-height: 1.8; white-space: pre-wrap;">
      <div>{old_text}<span style="color: red; font-weight: bold;">{added_text}</span></div>
    </div>
    """
    display(HTML(html))

    return new_input_ids

# 1回だけ進める
input_ids = step_once_for_class(input_ids, top_k=8)

In [ ]:
# @title 単語の生成を繰り返し行う
from transformers import TextIteratorStreamer
from threading import Thread
from IPython.display import display, HTML
import html
import time

@torch.no_grad()
def auto_write_stream_smooth(
    input_ids,
    max_new_words=60,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3,
    temperature=0.8,
    top_p=0.95,
    wait_time=0.05,
):
    start_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)

    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True
    )

    generation_kwargs = dict(
        input_ids=input_ids,
        max_new_tokens=max_new_words,
        repetition_penalty=repetition_penalty,
        no_repeat_ngram_size=no_repeat_ngram_size,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.eos_token_id,
        streamer=streamer,
    )

    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    generated_text = ""

    handle = display(HTML("<div style='font-size:20px;'></div>"), display_id=True)

    for new_piece in streamer:
        generated_text += new_piece

        start_html = html.escape(start_text)
        done_html = html.escape(generated_text[:-len(new_piece)] if len(new_piece) > 0 else generated_text)
        current_html = html.escape(new_piece)

        html_block = f"""
        <div style="font-size: 20px; line-height: 1.5; white-space: pre-wrap;">
          <div>{start_html}<span style="color: red;">{done_html}</span><span style="color: red; background-color: #ffe5e5; font-weight: bold;">{current_html}</span></div>
        </div>
        """

        handle.update(HTML(html_block))

        if wait_time > 0:
            time.sleep(wait_time)

    thread.join()

    final_text = start_text + generated_text

    final_html = f"""
    <div style="font-size: 20px; line-height: 1.5; white-space: pre-wrap;">
      <div>{html.escape(start_text)}<span style="color: red; font-weight: bold;">{html.escape(generated_text)}</span></div>
    </div>
    """
    handle.update(HTML(final_html))

    new_input_ids = tokenizer(final_text, return_tensors="pt").input_ids.to(model.device)
    return new_input_ids, final_text

# 実行
input_ids, final_text = auto_write_stream_smooth(input_ids, max_new_words=60, wait_time=0.05)

# 対話への応用

In [ ]:
# @title 対話をリセット
chat_history = []
time.sleep(0.5)
print("会話をリセットしました")

In [ ]:
# @title LLMによる対話と内部処理
from transformers import StoppingCriteria, StoppingCriteriaList
from transformers import TextIteratorStreamer
from threading import Thread
from IPython.display import display, HTML
import torch
import html
import time
import re

if "chat_history" not in globals():
    chat_history = []

USER_TAG = "[[USER]]"
ASSISTANT_TAG = "[[ASSISTANT]]"

def build_instruction_lines():
    return [
        "以下は、USERとASSISTANTの自然な日本語の会話です。",
        "ASSISTANTは優秀なAIアシスタントで、USERの発言に対して自然な日本語で短く返事します。",
        "話者は[[USER]]のような形で必ず書き、他のフォーマットを使用してはいけません。",
        "USERとASSISTANT以外の話者は一切登場しません。",
        "",
    ]

def build_chat_prompt_for_generation(user_message, chat_history):
    """
    実際にモデルへ渡す、生成直前/生成中の全文。
    最後は必ず [[ASSISTANT]] で終わる。
    """
    lines = build_instruction_lines()

    for turn in chat_history:
        lines.append(f"{USER_TAG} {turn['user']}")
        lines.append(f"{ASSISTANT_TAG} {turn['assistant']}")

    lines.append(f"{USER_TAG} {user_message}")
    lines.append(f"{ASSISTANT_TAG} ")

    return "\n".join(lines)

def build_chat_transcript(chat_history):
    """
    生成完了後の確定済み全文。
    存在しない次ターンの [[USER]] は作らない。
    """
    lines = build_instruction_lines()

    for turn in chat_history:
        lines.append(f"{USER_TAG} {turn['user']}")
        lines.append(f"{ASSISTANT_TAG} {turn['assistant']}")

    return "\n".join(lines).rstrip()

def render_actual_prompt_panel(prompt_text, generated_assistant_text=""):
    """
    実際の入力全文をそのまま表示する。
    既にモデルに与えられている部分は全部黒、
    今回新しく生成された部分だけ赤で表示する。
    """
    base_html = html.escape(prompt_text)

    if generated_assistant_text:
        base_html += f"<span style='color:#d93025; font-weight:bold; background:#ffe5e5;'>{html.escape(generated_assistant_text)}</span>"

    return (
        "<div style='font-size:16px; line-height:1.2; white-space:pre-wrap; font-family:monospace; "
        "background:#eaeaea; padding:10px; border-radius:8px; color:#000;'>"
        + base_html
        + "</div>"
    )

class StopOnSubsequence(StoppingCriteria):
    def __init__(self, tokenizer, stop_patterns, prompt_length):
        self.tokenizer = tokenizer
        self.stop_patterns = stop_patterns
        self.prompt_length = prompt_length

    def __call__(self, input_ids, scores, **kwargs):
        generated_ids = input_ids[0][self.prompt_length:]
        generated_text = self.tokenizer.decode(generated_ids, skip_special_tokens=True)
        lowered = generated_text.lower()
        return any(re.search(pattern, lowered, flags=re.IGNORECASE) for pattern in self.stop_patterns)

def cut_off_role_leak(text):
    """
    タグっぽいものや他の話者ラベルが出たらそこから先を切る
    """
    pos = text.find("[[")
    if pos != -1:
        text = text[:pos]

    patterns = [
        r"\n\s*私：",
        r"\n\s*あなた：",
        r"\n\s*我：",
        r"\n\s*你：",
    ]

    cut_positions = []
    for pattern in patterns:
        m = re.search(pattern, text)
        if m:
            cut_positions.append(m.start())

    if cut_positions:
        text = text[:min(cut_positions)]

    return text.strip()

def render_chat_panel(chat_history, current_user_message=None, current_assistant_text="", streaming=False):
    bubbles = []

    for turn in chat_history:
        # ユーザー（右） ← 絵文字を右に
        bubbles.append(f"""
<div style="display:flex; justify-content:flex-end; margin:4px 0;">
  <div style="max-width:70%; display:flex; align-items:flex-end; gap:6px;">
    <div style="
      background:#cfe8ff;
      color:#000;
      padding:8px 12px;
      border-radius:16px 16px 4px 16px;
      white-space:pre-wrap;
      word-break:break-word;
      box-shadow:0 1px 2px rgba(0,0,0,0.08);
    ">{html.escape(turn['user'])}</div>
    <div style="font-size:18px;">🙂</div>
  </div>
</div>
""")

        # AI（左）
        bubbles.append(f"""
<div style="display:flex; justify-content:flex-start; margin:4px 0;">
  <div style="max-width:70%; display:flex; align-items:flex-end; gap:6px;">
    <div style="font-size:18px;">🤖</div>
    <div style="
      background:#f1f3f4;
      color:#000;
      padding:8px 12px;
      border-radius:16px 16px 16px 4px;
      white-space:pre-wrap;
      word-break:break-word;
      box-shadow:0 1px 2px rgba(0,0,0,0.08);
    ">{html.escape(turn['assistant'])}</div>
  </div>
</div>
""")

    # 今回の入力
    if current_user_message is not None:
        bubbles.append(f"""
<div style="display:flex; justify-content:flex-end; margin:4px 0;">
  <div style="max-width:70%; display:flex; align-items:flex-end; gap:6px;">
    <div style="
      background:#cfe8ff;
      padding:8px 12px;
      border-radius:16px 16px 4px 16px;
      white-space:pre-wrap;
    ">{html.escape(current_user_message)}</div>
    <div style="font-size:18px;">🙂</div>
  </div>
</div>
""")

        if streaming:
            style = "color:#d93025;"
        else:
            style = "color:#d93025; font-weight:bold;"

        bubbles.append(f"""
<div style="display:flex; justify-content:flex-start; margin:4px 0;">
  <div style="max-width:70%; display:flex; align-items:flex-end; gap:6px;">
    <div style="font-size:18px;">🤖</div>
    <div style="
      background:#f1f3f4;
      {style}
      padding:8px 12px;
      border-radius:16px 16px 16px 4px;
      white-space:pre-wrap;
    ">{html.escape(current_assistant_text)}</div>
  </div>
</div>
""")

    return (
        "<div style='"
        "font-size:18px; line-height:1.2;"
        "background:#eaf7e9;"
        "padding:10px 12px;"
        "border-radius:12px;"
        "max-width:600px;"
        "margin-left:20px;"   # ← 左寄せに変更
        "'>"
        + "".join(bubbles)
        + "</div>"
    )

def render_full_html(
    chat_history,
    prompt_text,
    current_user_message=None,
    current_assistant_text="",
    streaming=False
):
    chat_panel = render_chat_panel(
        chat_history,
        current_user_message=current_user_message,
        current_assistant_text=current_assistant_text,
        streaming=streaming,
    )

    internal_panel = render_actual_prompt_panel(
        prompt_text=prompt_text,
        generated_assistant_text=current_assistant_text
    )

    return (
        "<div style='font-family:sans-serif;'>"
        "<div style='font-size:22px; font-weight:bold; margin-bottom:6px;'>💬 生成AIとの対話において我々が目にする画面</div>"
        f"{chat_panel}"
        "<div style='height:14px;'></div>"
        "<div style='font-size:22px; font-weight:bold; margin-bottom:6px;'>🔍 内部でAIが扱う文章（途中までの文章＋生成した続きの文章）</div>"
        f"{internal_panel}"
        "</div>"
    )

@torch.no_grad()
def chat_reply_stream(
    user_message,
    chat_history,
    max_new_words=60,
    repetition_penalty=1.15,
    no_repeat_ngram_size=3,
    temperature=0.8,
    top_p=0.95,
    wait_time=0.03,
    start_delay=1.0,   # 入力後に少し待ってから生成開始
):
    prompt = build_chat_prompt_for_generation(user_message, chat_history)
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    prompt_length = input_ids.shape[1]

    stop_patterns = [
        r"\[\[",
        r"\n\s*私：",
        r"\n\s*あなた：",
        r"\n\s*我：",
        r"\n\s*你：",
    ]

    stopping_criteria = StoppingCriteriaList([
        StopOnSubsequence(tokenizer, stop_patterns, prompt_length)
    ])

    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True
    )

    generation_kwargs = dict(
        input_ids=input_ids,
        max_new_tokens=max_new_words,
        repetition_penalty=repetition_penalty,
        no_repeat_ngram_size=no_repeat_ngram_size,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.eos_token_id,
        streamer=streamer,
        stopping_criteria=stopping_criteria,
    )

    # まず、まだ返答が始まっていない状態を表示
    handle = display(
        HTML(render_full_html(
            chat_history=chat_history,
            prompt_text=prompt,
            current_user_message=user_message,
            current_assistant_text="",
            streaming=True
        )),
        display_id=True
    )

    # 入力後、少し待ってから生成開始
    if start_delay > 0:
        time.sleep(start_delay)

    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    assistant_text = ""

    for new_piece in streamer:
        assistant_text += new_piece
        assistant_text = cut_off_role_leak(assistant_text)

        handle.update(HTML(
            render_full_html(
                chat_history=chat_history,
                prompt_text=prompt,
                current_user_message=user_message,
                current_assistant_text=assistant_text,
                streaming=True
            )
        ))

        if wait_time > 0:
            time.sleep(wait_time)

    thread.join()

    assistant_text = cut_off_role_leak(assistant_text)

    chat_history.append({
        "user": user_message,
        "assistant": assistant_text
    })

    # 生成完了後は、確定済み履歴だけの全文を表示する
    final_prompt = build_chat_transcript(chat_history)

    handle.update(HTML(render_full_html(
        chat_history=chat_history,
        prompt_text=prompt,
        current_user_message=None,
        current_assistant_text=assistant_text,
        streaming=False
    )))

    return assistant_text, chat_history

#####################

while True:
  user_message = input("入力してください（Enterで確定）：").strip()
  if user_message:
    break
print()

assistant_text, chat_history = chat_reply_stream(
    user_message,
    chat_history,
    max_new_words=60,
    wait_time=0.03,
    start_delay=1.0
)